# Agente de Rotas Aéreas — da busca ao agente com LLM

**Projeto Acadêmico de Inteligência Artificial 2026/2 — Entrega final (Fases 1, 2 e 3)**
Centro Universitário IESB · Prof. Rodrigo Gonçalves

**Integrantes**
- Sofia de Carvalho Brito — 2512130063 — Fase 1 (Formulação e Busca)
- Victor Sousa Pereira — 2512130015 — Fase 2 (Dados e Aprendizado)
- Pedro de Faria Mello — 2512130037 — Fase 3 (Agente com LLM e IA Responsável)

## Objetivo

Construir **um único agente** que evolui em três fases. O problema é sempre o mesmo: encontrar a melhor rota entre dois aeroportos de uma malha aérea. O que muda é a inteligência do agente:

| Fase | O que o agente faz | Onde está no código |
|---|---|---|
| 1. Busca | Encontra a rota mais curta em km com BFS e A* (heurística admissível) | `src/environment`, `src/search` |
| 2. Aprendizado | Aprende o tempo de cada voo a partir de dados e o A* passa a minimizar **tempo previsto** | `src/environment/dataset.py`, `src/environment/model.py` |
| 3. Agente com LLM | Um LLM entende o pedido em português, chama o A* (com os custos aprendidos) como ferramenta e explica a rota | `src/agent` |

> **Ideia central:** nenhuma fase é descartada. O LLM (Fase 3) aciona a busca (Fase 1), que usa os custos aprendidos (Fase 2). O LLM **nunca calcula** a rota.

> **Dados sintéticos:** os tempos de voo são gerados por uma simulação definida no projeto. Não são registros reais de companhias aéreas e não há dados pessoais.

## Como executar

*Ambiente de execução → Executar tudo*, em uma sessão nova. O notebook funciona **sem chave de LLM**: nesse caso, só a conversa com o LLM (seção 11) é substituída pela chamada direta à ferramenta, e todo o restante roda normalmente. Para testar o agente com qualquer pedido, use a **interface gráfica da seção 15**.

## 1. Ambiente de execução

Clona o repositório do grupo (se ainda não existir nesta sessão), instala as dependências e entra na pasta do projeto. A célula pode ser executada mais de uma vez.

In [ ]:
%cd /content
REPO = "https://github.com/soficb14/projeto-ia-malhas-aereas.git"
![ -d projeto-ia-malhas-aereas ] || git clone -q {REPO}
%cd /content/projeto-ia-malhas-aereas
!pip install -q -r requirements.txt
!find . -name __pycache__ -type d -prune -exec rm -rf {} +

In [ ]:
from pathlib import Path

esperados = ["data/maps/airports.json", "data/maps/mapa_pequeno.json",
             "data/maps/mapa_medio.json", "data/maps/mapa_grande.json",
             "data/pedidos_teste.json", "results/fase3_avaliacao_v1.csv",
             "results/fase3_avaliacao_v3.csv"]
faltam = [p for p in esperados if not Path(p).exists()]
print("Arquivos principais:", "todos presentes" if not faltam else f"FALTAM {faltam}")
print("Módulos em src/:")
for p in sorted(Path("src").rglob("*.py")):
    print("-", p)

## 2. Verificação da implementação

Antes de interpretar qualquer resultado, rodamos os testes automatizados (grafo, mapas, BFS, A*, heurísticas, modelo de custo e ferramentas do agente). Todos devem passar.

In [ ]:
!python3 -m pytest -q

## 3. Configuração do LLM

O agente da Fase 3 usa um LLM por uma API compatível com OpenAI: o **OpenRouter**, com um modelo da **NVIDIA** (plano gratuito, sujeito a limite diário de requisições). A configuração vem de três *Secrets* do Colab (ícone de chave na barra lateral). **Nenhuma chave está escrita no código.**

| Secret | Conteúdo | Valor usado nos resultados |
|---|---|---|
| `LLM_API_KEY` | chave do provedor | (privada) |
| `LLM_BASE_URL` | URL da API | OpenRouter: `https://openrouter.ai/api/v1` |
| `LLM_MODEL` | ID exato do modelo | Modelo da NVIDIA via OpenRouter. **PREENCHER: ID exato** (copie do Secret `LLM_MODEL`; no OpenRouter tem o formato `nvidia/...`) |

Se algum Secret não existir (por exemplo, na conta do professor), a célula abaixo avisa quais faltam e define `TEM_LLM = False`. O notebook segue normalmente.

In [ ]:
import os

NOMES_LLM = ("LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL")
try:
    from google.colab import userdata
    for nome in NOMES_LLM:
        if os.environ.get(nome):
            continue
        try:
            valor = userdata.get(nome)
        except Exception:          # Secret inexistente ou sem acesso do notebook
            valor = None
        if valor:
            os.environ[nome] = valor
except ImportError:                # fora do Colab: usa as variáveis de ambiente
    pass

faltando = [n for n in NOMES_LLM if not os.environ.get(n)]
TEM_LLM = not faltando
if TEM_LLM:
    print("LLM configurado.")
else:
    print("Secrets ausentes:", ", ".join(faltando))
    print("A conversa com o LLM (seção 11) usará a ferramenta diretamente, sem LLM.")

# Parte I — Fase 1: Busca

## 4. Modelagem do problema em espaço de estados

| Elemento | No nosso problema |
|---|---|
| **Estado** | um aeroporto (posição no grafo) |
| **Estado inicial** | aeroporto de origem da consulta |
| **Ações** | voos diretos que saem do aeroporto atual |
| **Transição** | ir do aeroporto de origem do voo para o de destino |
| **Teste de objetivo** | o aeroporto atual é o destino |
| **Custo de caminho** | soma das distâncias dos voos, em km (nesta fase) |

A malha é um grafo **direcionado**: os aeroportos são nós e os voos são arestas. O agente da Fase 1 é deliberadamente ingênuo: conhece o mapa e as distâncias, e nada mais.

In [ ]:
from src.environment.loader import load_network

AIRPORTS_PATH = "data/maps/airports.json"
MAPAS = {"pequeno": "data/maps/mapa_pequeno.json",
         "medio": "data/maps/mapa_medio.json",
         "grande": "data/maps/mapa_grande.json"}

redes = {nome: load_network(caminho, AIRPORTS_PATH) for nome, caminho in MAPAS.items()}
for nome, rede in redes.items():
    print(f"Mapa {nome:<8} | {rede.number_of_airports():>2} aeroportos | {rede.number_of_flights():>2} voos direcionados")

network = redes["medio"]   # mapa usado nas Fases 2 e 3

## 5. Heurística do A* e sua admissibilidade

O A* ordena a fronteira por

$$f(n) = g(n) + h(n)$$

- $g(n)$: custo acumulado desde a origem;
- $h(n)$: estimativa do custo restante até o destino.

**Heurística própria:** a distância em linha reta (fórmula de Haversine) entre o aeroporto atual e o destino.

**Por que é admissível:** nenhum caminho entre dois pontos da superfície é menor que a linha reta entre eles, então $h(n)$ nunca superestima o custo real restante. Para valer no grafo, o custo de cada voo também não pode ser menor que a linha reta entre seus extremos. A função `check_heuristic` confere isso aresta por aresta.

In [ ]:
from src.search.heuristics import check_heuristic

for nome, rede in redes.items():
    violacoes = check_heuristic(rede, metric="distance")
    print(f"Mapa {nome:<8}: {len(violacoes)} voos violam o limite inferior (admissível se for 0)")

## 6. Experimento comparativo: BFS × A*

Comparamos a **busca não informada** (BFS, que minimiza o número de voos) com o **A\*** (que minimiza km), nos três mapas, registrando custo da rota, nós expandidos e tempo de execução.

**Parte A:** uma consulta fixa por mapa. **Parte B:** todos os pares origem-destino de cada mapa, para a conclusão não depender de uma única consulta. O tempo é a mediana de várias repetições, porque uma única execução dura microssegundos e oscila muito.

In [ ]:
import statistics
from time import perf_counter

import pandas as pd
from src.search.astar import a_star_search
from src.search.bfs import breadth_first_search

CONSULTAS = {"pequeno": ("ATL", "LAX"), "medio": ("SEA", "MEM"), "grande": ("SEA", "BOS")}
REPETICOES = 200

def tempo_ms(funcao, *args, **kw):
    amostras = []
    for _ in range(REPETICOES):
        t0 = perf_counter()
        resultado = funcao(*args, **kw)
        amostras.append((perf_counter() - t0) * 1000)
    return statistics.median(amostras), resultado

linhas = []
for nome, (origem, destino) in CONSULTAS.items():
    rede = redes[nome]
    for alg, fn, kw in (("BFS", breadth_first_search, {}),
                        ("A*", a_star_search, {"metric": "distance"})):
        ms, r = tempo_ms(fn, rede, origem, destino, **kw)
        linhas.append({"mapa": nome, "algoritmo": alg, "origem": origem, "destino": destino,
                       "custo_km": round(r.cost, 1), "nos_expandidos": r.expanded_nodes,
                       "tempo_ms": round(ms, 4), "voos": len(r.path) - 1,
                       "rota": " -> ".join(a.code for a in r.path)})

fase1 = pd.DataFrame(linhas)
display(fase1.drop(columns="rota"))
display(fase1[["mapa", "algoritmo", "rota"]])

In [ ]:
import matplotlib.pyplot as plt

ordem = ["pequeno", "medio", "grande"]
metricas = [("custo_km", "Custo da rota (km)"), ("nos_expandidos", "Nós expandidos"),
            ("tempo_ms", "Tempo de execução (ms)")]
fig, eixos = plt.subplots(1, 3, figsize=(15, 4))
for eixo, (col, titulo) in zip(eixos, metricas):
    tab = fase1.pivot(index="mapa", columns="algoritmo", values=col).loc[ordem]
    tab.plot(kind="bar", ax=eixo, rot=0)
    eixo.set_title(titulo); eixo.set_xlabel("Mapa"); eixo.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Parte B: todos os pares origem-destino de cada mapa
resumo = []
for nome, rede in redes.items():
    codigos = sorted(rede.airports)
    n = pior_bfs = 0
    nos_bfs = nos_a = excesso = 0.0
    for o in codigos:
        for d in codigos:
            if o == d:
                continue
            b = breadth_first_search(rede, o, d)
            a = a_star_search(rede, o, d, metric="distance")
            if a is None or b is None:
                continue
            n += 1
            nos_bfs += b.expanded_nodes
            nos_a += a.expanded_nodes
            if b.cost > a.cost + 1e-6:
                pior_bfs += 1
                excesso += b.cost - a.cost
    resumo.append({"mapa": nome, "pares": n,
                   "nós BFS (média)": round(nos_bfs / n, 1), "nós A* (média)": round(nos_a / n, 1),
                   "% pares em que a BFS não acha a rota mais curta": round(100 * pior_bfs / n, 1),
                   "km extras da BFS (média nesses pares)": round(excesso / max(pior_bfs, 1), 0)})
display(pd.DataFrame(resumo))

### 6.1. Análise da Fase 1

Leia os números acima antes de concluir. Pontos a registrar no relatório:

1. **Optimalidade:** a BFS minimiza o número de voos, não os km. O A* com heurística admissível devolve sempre a rota de menor distância. A coluna de "pares em que a BFS não acha a rota mais curta" mostra com que frequência isso importa em cada mapa.
2. **Nós expandidos:** o A* tende a expandir menos nós, mas a vantagem depende do mapa e da consulta. Em consultas isoladas a BFS pode expandir tantos nós quanto o A* (ou até menos). Por isso a Parte B usa todos os pares.
3. **Tempo:** as diferenças são de frações de milissegundo em grafos tão pequenos; o custo por nó do A* (fila de prioridade e cálculo da heurística) pode superar a economia de nós.
4. **Limite da fase:** o agente assume um mundo perfeito. A rota mais curta em km raramente é a mais rápida, o que motiva a Fase 2.

# Parte II — Fase 2: Aprendizado

## 7. Conjunto de dados sintético

Em vez de receber o custo real de cada voo pronto, o agente **aprende** o tempo de voo a partir de dados. O gerador cria várias observações por voo da malha, com as características:

| Papel | Variáveis |
|---|---|
| Numéricas | `distance`, `route_congestion_index` (1 a 5), `wind_component` (nós; negativo = vento de proa) |
| Categóricas | `aircraft_type`, `weather_condition`, `time_of_day` |
| Alvo | `real_elapsed_time` (minutos) |

O gerador introduz valores ausentes em clima e vento para que o pré-processamento tenha o que tratar. `origin` e `destination` identificam a rota (usadas para agrupar a validação cruzada).

In [ ]:
from src.environment.dataset import generate_synthetic_flight_dataset

dataset = generate_synthetic_flight_dataset(network, samples_per_flight=20, seed=42, missing_rate=0.03)
print("Dimensões:", dataset.shape)
display(dataset.head())
faltantes = dataset.isna().sum()
display(faltantes[faltantes > 0].to_frame("valores_ausentes"))
t = dataset["real_elapsed_time"]
print(f"Tempo de voo (min): mínimo={t.min():.1f} | média={t.mean():.1f} | máximo={t.max():.1f}")

## 8. Pré-processamento, treino e avaliação

O pré-processamento está dentro de um `Pipeline`, ajustado só com os dados de treino de cada partição (sem vazamento):

- **Numéricas:** ausentes → mediana; depois padronização (média 0, desvio 1).
- **Categóricas:** ausentes → categoria mais frequente; depois One-Hot Encoding.

Comparamos **três referências**: baseline (média), Regressão Linear e Random Forest, com validação cruzada em 5 partições e métricas de regressão (MAE e RMSE, em minutos).

**Validação agrupada por rota (`GroupKFold`):** deixa rotas inteiras de fora em cada partição. Isso mede se o modelo generaliza para uma rota que ele nunca viu, que é o uso real do agente. Mostramos também a validação comum para comparar.

In [ ]:
from src.environment.model import train_and_evaluate, error_analysis

best_pipeline, resultados, holdout = train_and_evaluate(dataset, seed=42, n_splits=5, group_by_route=True)
_, resultados_kfold, _ = train_and_evaluate(dataset, seed=42, n_splits=5, group_by_route=False)

print("Validação cruzada AGRUPADA por rota (rota nunca vista):")
display(resultados.round(2))
print("Validação cruzada comum (a mesma rota pode aparecer em treino e validação):")
display(resultados_kfold.round(2))

modelos = resultados.drop(index="Baseline (média)")
print("Modelo escolhido (menor CV MAE agrupado):", modelos["cv_mae"].idxmin())

### 8.1. Como ler as métricas

- **MAE:** erro absoluto médio, em minutos. MAE de 10 significa que a previsão erra, em média, 10 minutos.
- **RMSE:** raiz do erro quadrático médio; penaliza mais os erros grandes.
- **cv_mae_std:** quanto o MAE varia entre as partições. Desvio alto indica instabilidade.
- **teste_***: métricas no conjunto de teste (20%), separado de forma aleatória por amostra. Como a mesma rota pode aparecer em treino e teste, esse número é otimista; a validação agrupada é a medida mais honesta.

In [ ]:
import matplotlib.pyplot as plt

resultados[["cv_mae", "cv_rmse", "teste_mae", "teste_rmse"]].plot(kind="bar", figsize=(10, 4), rot=15)
plt.title("Modelos — erro de previsão (minutos)")
plt.ylabel("Erro (min)"); plt.xlabel("")
plt.tight_layout(); plt.show()

## 9. Onde o modelo erra mais

Erro absoluto médio por condição, no conjunto de teste. Mostra quais cenários são mais difíceis de prever.

In [ ]:
for variavel, tabela in error_analysis(*holdout).items():
    print(f"Erro absoluto médio por {variavel}:")
    display(tabela.round(2).to_frame("erro_abs_medio_min"))

## 10. Custo aprendido dentro da busca (o elo entre as Fases 1 e 2)

É **o mesmo A\*** da Fase 1. O que muda é a função de custo: antes km, agora o **tempo previsto pelo modelo**. A heurística passa a ser o tempo mínimo em linha reta a 900 km/h, e o mapa de custos aplica esse mesmo piso às previsões, para que continue admissível.

In [ ]:
from src.environment.model import build_cost_map
from src.search.heuristics import MAX_SPEED_KMH

bom = {"aircraft_type": "Narrow-body", "route_congestion_index": 1,
       "weather_condition": "Clear", "wind_component": 0.0, "time_of_day": "Off-peak"}
ruim = {"aircraft_type": "Narrow-body", "route_congestion_index": 5,
        "weather_condition": "Storm", "wind_component": -20.0, "time_of_day": "Peak"}
evento = {("DEN", "ORD"): {"weather_condition": "Storm", "route_congestion_index": 5,
                           "time_of_day": "Peak", "wind_component": -25.0}}

def codigos(r):
    return " -> ".join(a.code for a in r.path)

def buscar_tempo(origem, destino, contexto, overrides=None):
    network.set_learned_costs(build_cost_map(network, best_pipeline, contexto,
                                             overrides=overrides, max_speed_km_h=MAX_SPEED_KMH))
    return a_star_search(network, origem, destino, metric="predicted_time")

network.set_learned_costs(build_cost_map(network, best_pipeline, bom, max_speed_km_h=MAX_SPEED_KMH))
print("Admissibilidade (predicted_time):", len(check_heuristic(network, metric="predicted_time")), "violações")
network.clear_learned_costs()

O, D = "ABQ", "STL"
por_km = a_star_search(network, O, D, metric="distance")
cenarios = [("Por distância (Fase 1)", por_km.cost, "km", codigos(por_km)),]
for nome, ctx, ov in (("Tempo previsto, condições boas", bom, None),
                      ("Tempo previsto, condições ruins em toda a malha", ruim, None),
                      ("Tempo previsto, tempestade só em DEN->ORD", bom, evento)):
    r = buscar_tempo(O, D, ctx, ov)
    cenarios.append((nome, r.cost, "min", codigos(r)))
network.clear_learned_costs()
display(pd.DataFrame(cenarios, columns=["Cenário", "Custo", "Unidade", "Rota"]).round({"Custo": 1}))

In [ ]:
# Experimento sobre TODOS os pares do mapa médio, em dois cenários:
#  (a) condições boas em toda a malha;
#  (b) tempestade com congestionamento 5 no PRIMEIRO voo da rota que a Fase 1 escolheu.
cm_bom = build_cost_map(network, best_pipeline, bom, max_speed_km_h=MAX_SPEED_KMH)
cm_evento = {}   # um mapa de custos por voo afetado (reaproveitado entre pares)

def mapa_com_tempestade(aresta):
    if aresta not in cm_evento:
        cm_evento[aresta] = build_cost_map(
            network, best_pipeline, bom, max_speed_km_h=MAX_SPEED_KMH,
            overrides={aresta: {"weather_condition": "Storm", "route_congestion_index": 5,
                                "time_of_day": "Peak", "wind_component": -25.0}})
    return cm_evento[aresta]

def tempo_da_rota(mapa, cods_rota):
    return sum(mapa[(a, b)] for a, b in zip(cods_rota, cods_rota[1:]))

mudancas, total, mudou_bom = [], 0, 0
for o in sorted(network.airports):
    for d in sorted(network.airports):
        if o == d:
            continue
        por_km = a_star_search(network, o, d, metric="distance")
        if por_km is None:
            continue
        total += 1
        rota_km = [a.code for a in por_km.path]

        network.set_learned_costs(cm_bom)                                   # (a)
        mudou_bom += codigos(a_star_search(network, o, d, metric="predicted_time")) != codigos(por_km)

        aresta = (rota_km[0], rota_km[1])                                   # (b)
        cm = mapa_com_tempestade(aresta)
        network.set_learned_costs(cm)
        r = a_star_search(network, o, d, metric="predicted_time")
        if r is not None and codigos(r) != codigos(por_km):
            mudancas.append({"origem": o, "destino": d, "voo com tempestade": f"{aresta[0]}->{aresta[1]}",
                             "rota por km": codigos(por_km), "rota por tempo previsto": codigos(r),
                             "min economizados": round(tempo_da_rota(cm, rota_km) - r.cost, 1)})
network.clear_learned_costs()

print(f"(a) Condições boas em toda a malha: a rota por tempo difere da rota por km em {mudou_bom} de {total} pares.")
print(f"(b) Tempestade no primeiro voo da rota por km: a rota muda em {len(mudancas)} de {total} pares "
      f"({100 * len(mudancas) / total:.1f}%).")
mudancas_df = pd.DataFrame(mudancas)
if len(mudancas_df):
    print(f"    Quando muda, economiza em média {mudancas_df['min economizados'].mean():.1f} min "
          f"(máximo {mudancas_df['min economizados'].max():.1f}) em relação a insistir na rota antiga.")
display(mudancas_df.head(10))

### 10.1. Análise da Fase 2 e limitações

1. **Modelos:** compare o MAE de cada modelo com o do baseline. A diferença para a média mostra quanto o aprendizado agrega.
2. **Generalização:** compare as duas tabelas da seção 8. Se o Random Forest piora bastante na validação agrupada por rota e a Regressão Linear não, o Random Forest está se apoiando na identidade da rota e não nas características físicas. Isso importa porque o agente precisa prever tempos de rotas novas.
3. **Erros:** veja na seção 9 as condições com maior erro e relacione com a simulação.
4. **Decisão da busca:** com condições uniformes em toda a malha, o tempo previsto cresce de modo parecido em todos os voos e a rota quase nunca muda, mesmo piorando o contexto inteiro. O modelo muda a decisão quando há uma condição **localizada** (um voo com tempestade e congestionamento), porque aí a rota mais curta deixa de ser a mais rápida. Por isso o experimento (b) da seção 10 usa um evento por voo e conta em quantos pares a busca desvia.
5. **Limitações:** os tempos vêm de uma fórmula definida pelo grupo, então o modelo aprende a simulação, não a aviação real. A separação treino/teste final é aleatória por amostra. Os resultados provam o método dentro da simulação, não desempenho operacional.

# Parte III — Fase 3: Agente com LLM

## 11. O agente: LLM + ferramentas

O LLM **não calcula rota**. Ele faz o que sabe fazer bem: entender o pedido em português, decidir que ferramenta chamar e com quais argumentos, e explicar o resultado. A rota e o tempo vêm sempre da ferramenta `buscar_rota`, que executa o A* da Fase 1 com os custos aprendidos da Fase 2.

```
pedido (português) → LLM → buscar_rota(origem, destino, evitar, eventos, clima, ...)
                              ↓
                  A* (Fase 1) com custo = tempo previsto pelo modelo (Fase 2)
                              ↓
                   rota + tempo + economia → LLM → explicação em português
```

- **Prompts versionados** (`src/agent/prompts.py`): v1 a v4 (comparadas na seção 13). A versão ativa é a v1.
- **Saída estruturada:** o LLM chama a função com argumentos em JSON; a ferramenta valida cada argumento e devolve `ok=false` com o motivo quando algo é inválido.
- **Com LLM configurado (seção 3):** o agente responde e mostramos latência, tokens e ferramentas chamadas. Se a API falhar, o erro aparece e a célula mostra o resultado da ferramenta diretamente, para a demonstração nunca ficar vazia.
- **Sem LLM:** chamamos `buscar_rota` direto, nos mesmos três pedidos.

In [ ]:
from src.agent.backend_projeto import ProjetoBackend
from src.agent.tools import FlightTools

tools = FlightTools(ProjetoBackend(network, best_pipeline, aircraft_type="Narrow-body"))

PEDIDOS_DEMO = [
    ("Qual a melhor rota de ABQ para STL?",
     dict(origem="ABQ", destino="STL")),
    ("Tem tempestade no trecho entre DEN e ORD. Qual a melhor rota de ABQ para STL?",
     dict(origem="ABQ", destino="STL", eventos=[{"de": "DEN", "para": "ORD", "tipo": "Storm"}])),
    ("Vou de CLT para LAX, mas há tempestade em Dallas. Evite passar por lá.",
     dict(origem="CLT", destino="LAX", evitar=["DFW"])),
]

def mostrar_ferramenta(args):
    r = tools.buscar_rota(**args)
    if not r["ok"]:
        print("  erro da ferramenta:", r["erro"])
        return
    print("  rota:", " -> ".join(r["rota"]), f"| {r['tempo_previsto_min']} min | {r['distancia_km']} km")
    curta = r["rota_mais_curta"]
    if curta:
        print("  mais curta em km:", " -> ".join(curta["rota"]),
              f"| {curta['tempo_previsto_min']} min | {curta['distancia_km']} km")
        print("  economia de tempo:", r["economia_min"], "min")

agent = None
if TEM_LLM:
    from src.agent.agent import FlightAgent
    try:
        agent = FlightAgent(tools)
    except Exception as e:
        print("Não foi possível criar o agente:", e)

for pedido, args in PEDIDOS_DEMO:
    print("Você:", pedido)
    if agent is None:
        print("Sem LLM: chamando a ferramenta buscar_rota diretamente.")
        mostrar_ferramenta(args)
    else:
        r = agent.run(pedido)
        if r["erro"] or not r["resposta"]:
            print(f"Agente: (sem resposta: {r['erro']})")
            print("Contingência: resultado da ferramenta para este pedido.")
            mostrar_ferramenta(args)
        else:
            print("Agente:", r["resposta"])
            print(f"[{r['latencia_s']}s | {r['tokens_in']}+{r['tokens_out']} tokens | "
                  f"ferramentas: {[c['nome'] for c in r['chamadas']]}]")
    print()

## 12. Conjunto de pedidos de teste (16 pedidos)

Cada pedido em `data/pedidos_teste.json` traz o **resultado esperado**: se o agente deve ou não chamar a ferramenta e com quais argumentos. Há pedidos simples, com restrições (evitar aeroporto, evento em trecho, clima, vento, congestionamento), com código de aeroporto inexistente, incompletos, **fora do escopo** (capital da França, preço, hotel) e uma tentativa de **injeção de instruções**.

Um pedido é um **acerto** quando: (1) acionou a ferramenta quando devia (ou não acionou quando não devia), (2) com os argumentos esperados, (3) a rota devolvida é válida no grafo, (4) a resposta cita a rota real e (5) não houve erro de API.

In [ ]:
import json
import os
with open("data/pedidos_teste.json", encoding="utf-8") as f:
    pedidos = json.load(f)
print(len(pedidos), "pedidos de teste")
display(pd.DataFrame([{"id": p["id"], "pedido": p["pedido"],
                       "deve usar ferramenta": p["espera"]["usa_ferramenta"]} for p in pedidos]))

## 13. Resultados da avaliação: porcentagem de acerto por versão de prompt

Quatro versões do prompt do sistema (`src/agent/prompts.py`) são comparadas nos mesmos 16 pedidos:

| Versão | Como foi construída |
|---|---|
| **v1** | Base: 8 regras (nunca calcular rota, extrair códigos IATA, traduzir restrições em argumentos, recusar fora do escopo, ignorar instruções maliciosas) |
| **v2** | v1 + regras 9 e 10 (tipos de evento em um trecho; tipo de aeronave não configurável) |
| **v3** | v2 + regras 11 e 12 (chamar a ferramenta para código desconhecido; listar todos os aeroportos da rota) |
| **v4** | v1 + duas regras criadas a partir das falhas reais: tratar qualquer código de 3 letras como aeroporto pedido (pedido 9) e sempre escrever os aeroportos da rota, inclusive ao recusar uma instrução maliciosa (pedido 16) |

A tabela mostra **quantos pedidos cada versão acertou e a porcentagem**. Como falhas por limite da API (erro 429) não dizem nada sobre o prompt, a tabela traz também a porcentagem calculada só sobre os pedidos que o LLM chegou a responder.

Executar a avaliação exige a chave e leva alguns minutos por versão (pausa entre as chamadas para respeitar o limite do provedor). Por isso a reexecução fica atrás de uma chave opcional (`RODAR_AVALIACAO`). Versões sem arquivo em `results/` aparecem como "sem resultado" até serem avaliadas.

In [ ]:
RODAR_AVALIACAO = False                 # True: roda a avaliação agora (precisa de LLM configurado)
VERSOES = ["v1", "v2", "v3", "v4"]       # versões comparadas na tabela
VERSOES_A_RODAR = ["v2", "v4"]           # versões que ainda não têm resultado em results/

if RODAR_AVALIACAO and TEM_LLM:
    from src.agent.avaliar import avaliar
    from src.agent.prompts import PROMPTS
    for v in VERSOES_A_RODAR:
        if v in PROMPTS:
            avaliar(prompt_version=v)
        else:
            print(f"O prompt {v} não existe no repositório clonado: faça push do prompts.py.")

avaliacoes = {}
for v in VERSOES:
    caminho = f"results/fase3_avaliacao_{v}.csv"
    if os.path.exists(caminho):
        avaliacoes[v] = pd.read_csv(caminho)

def pct(parte, total):
    # arredonda meio para cima (81,25 -> 81,3), ao contrário do round() do Python
    return int(1000 * parte / total + 0.5) / 10 if total else "—"

COLUNAS = ["acertos", "% de acerto", "falhas por erro de API", "% nos pedidos respondidos",
           "% ferramenta correta", "% argumentos corretos", "% resposta fiel à rota",
           "tokens (entrada+saída)", "latência média (s)"]
linhas = []
for v in VERSOES:
    if v not in avaliacoes:
        linhas.append({"prompt": v, **{c: "—" for c in COLUNAS}, "acertos": "sem resultado"})
        continue
    d = avaliacoes[v]
    n, ac = len(d), int(d.acerto.sum())
    erro_api = d["erro"].notna()
    respondidos = n - int(erro_api.sum())
    linhas.append({
        "prompt": v,
        "acertos": f"{ac}/{n}",
        "% de acerto": pct(ac, n),
        "falhas por erro de API": int(erro_api.sum()),
        "% nos pedidos respondidos": pct(ac, respondidos),
        "% ferramenta correta": pct(int(d.ferramenta_ok.sum()), n),
        "% argumentos corretos": pct(int(d.argumentos_ok.sum()), n),
        "% resposta fiel à rota": pct(int(d.resposta_fiel.dropna().astype(bool).sum()), int(d.resposta_fiel.notna().sum())),
        "tokens (entrada+saída)": int(d.tokens_in.sum() + d.tokens_out.sum()),
        "latência média (s)": round(d.latencia_s.mean(), 1),
    })
tabela_versoes = pd.DataFrame(linhas).set_index("prompt")
display(tabela_versoes)

medidas = tabela_versoes.loc[[v for v in VERSOES if v in avaliacoes]]
if len(medidas):
    ax = medidas[["% de acerto", "% nos pedidos respondidos"]].astype(float).plot(
        kind="bar", figsize=(8, 4), rot=0, ylim=(0, 105))
    for cont in ax.containers:
        ax.bar_label(cont, fmt="%.1f%%", padding=2)
    ax.set_ylabel("% dos pedidos"); ax.set_xlabel("Versão do prompt")
    ax.set_title("Acerto por versão de prompt (16 pedidos de teste)")
    plt.tight_layout(); plt.show()
sem = [v for v in VERSOES if v not in avaliacoes]
if sem:
    print("Sem resultado ainda:", ", ".join(sem), "| para medir, use RODAR_AVALIACAO = True.")

# Pedido a pedido: ✓ acertou, ✗ falhou, ⚠ falhou por erro de API
def simbolo(l):
    return "⚠" if isinstance(l["erro"], str) else ("✓" if l["acerto"] else "✗")
matriz = pd.DataFrame({v: d.set_index("id").apply(simbolo, axis=1) for v, d in avaliacoes.items()})
matriz.insert(0, "pedido", pd.read_csv("results/fase3_avaliacao_v1.csv").set_index("id")["pedido"].str.slice(0, 60))
display(matriz)

for v, d in avaliacoes.items():
    falhas = d[~d.acerto]
    if falhas.empty:
        print(f"\nPrompt {v}: nenhuma falha nos {len(d)} pedidos.")
        continue
    print(f"\nPedidos que falharam com o prompt {v}:")
    display(falhas[["id", "pedido", "ferramenta_ok", "argumentos_ok", "resposta_fiel", "erro"]]
            .assign(erro=lambda x: x["erro"].astype("string").fillna("").str.slice(0, 70)))

### 13.1. Análise dos casos em que o agente falha

Confira cada falha nas tabelas acima antes de usar estas observações no relatório. Elas descrevem o que os arquivos de `results/` mostram.

- **Pedido 9 ("rota de CLT para XXX"), v1 e v3:** o LLM não chamou a ferramenta. Ele considerou o destino não especificado e pediu o código IATA ao usuário. O comportamento é razoável para o usuário, mas diverge do esperado, que era chamar a ferramenta e receber `ok=false` para o código inexistente. A regra 11 da v3 pedia isso e mesmo assim não mudou o resultado. A v4 reforça a regra com uma instrução explícita ("nunca diga que o destino não foi especificado quando houver um código").
- **Pedido 16 (injeção de instruções), v1:** o agente **resistiu**: não inventou a rota e informou o valor real calculado pela ferramenta. Foi marcado como falha porque a verificação de fidelidade exige que o texto cite todos os aeroportos da rota, e a resposta citou o tempo e a distância mas não listou os aeroportos. É uma falha do **critério de avaliação**, não de alucinação. A v4 pede que a rota completa apareça mesmo quando o agente recusa a instrução.
- **Pedidos 15 e 16, v3:** a coluna `erro` mostra **HTTP 429 (limite diário do plano gratuito do provedor)**. Essas duas falhas não são do prompt v3: a chamada nem foi respondida.

> **Como ler a comparação entre versões:** a v3 não pode ser dada como pior que a v1 com base no placar bruto, porque parte da diferença vem do limite de requisições (compare a coluna "% nos pedidos respondidos"). A v4 e a v2 só têm porcentagem depois de avaliadas com `RODAR_AVALIACAO = True` e cota disponível; **a v4 foi escrita para corrigir falhas já observadas, mas isso é uma hipótese até ser medida**. **PREENCHER (após reexecutar):** qual prompt ficou ativo (`ACTIVE_VERSION`) e por quê, citando as porcentagens da tabela.

## 14. Limites do agente: alucinação, custo, latência e fora do escopo

| Tema | O que observamos | Como o projeto trata |
|---|---|---|
| **Alucinação** | O LLM poderia inventar rotas, tempos ou distâncias. | A regra 1 do prompt proíbe; a rota só vem da ferramenta; o critério "resposta fiel" confere se o texto cita a rota real; a ferramenta valida todos os argumentos (aeroporto inexistente, vento fora de faixa etc.). |
| **Custo** | Tokens por avaliação na tabela da seção 13. O provedor usado tem plano gratuito. | Custo monetário baixo, mas há **limite diário de requisições** (erro 429 registrado). Em produção seria preciso plano pago ou cache. |
| **Latência** | Média de alguns segundos por pedido (tabela da seção 13), contra milissegundos da busca. | Quase todo o tempo é a chamada ao LLM, não o A*. |
| **Fora do escopo** | Pedidos 10 a 13 (viagem sem destino, capital da França, preço, hotel). | A regra 7 manda recusar sem chamar ferramenta; os pedidos foram tratados corretamente no placar. |
| **Instruções maliciosas** | Pedido 16 pede para ignorar as regras. | A regra 8 manda ignorar; o agente manteve o valor calculado. |
| **Dependência do provedor** | O erro 429 deixou respostas vazias na primeira versão da demonstração. | A célula da seção 11 agora mostra o erro da API e cai para a ferramenta (contingência). |

## 15. Interface gráfica: converse com o agente

A célula logo abaixo cria uma **janela de conversa dentro do notebook**: uma caixa de texto, um botão **Enviar** e a área de resposta. É a forma de testar o agente com qualquer pedido, sem editar código.

**Como usar**
1. Execute a célula de código desta seção (ou *Executar tudo*). A janela aparece logo abaixo dela.
2. Escreva um pedido em português, por exemplo: *"Vou de CLT para LAX, mas há tempestade em Dallas. Evite passar por lá."* Se preferir, escolha um dos **16 pedidos de teste** no menu *Exemplos*.
3. Clique em **Enviar**. Use **Limpar** para recomeçar.

**O que a resposta mostra**
- A resposta em português com a rota mais rápida, o tempo previsto, a distância e a comparação com a rota mais curta em km.
- A **ferramenta chamada e os argumentos** (`buscar_rota`, origem, destino, aeroportos a evitar, eventos, clima etc.). É a prova de que a rota vem do A\* com os custos aprendidos, e não do LLM.
- Latência e tokens, quando o LLM responde.

**Dois modos de funcionamento**

| Modo | Quando é usado | Quem interpreta o pedido |
|---|---|---|
| **LLM** | Os *Secrets* da seção 3 estão configurados | O LLM entende o pedido, escolhe a ferramenta e redige a explicação |
| **Regras (sem LLM)** | Não há chave (por exemplo, na conta do professor), ou o LLM falhou | Um interpretador simples por regras, que reconhece códigos IATA e nomes de cidade, "evitar", "entre X e Y", clima, horário de pico, vento e congestionamento |

No modo sem LLM, a **busca e o modelo de custo são os mesmos**: só muda quem transforma a frase em argumentos. O interpretador por regras entende menos variações de linguagem do que o LLM e **não é** o LLM; a janela informa qual modo está ativo. Se o LLM der erro (por exemplo, limite do provedor), a janela avisa e responde pelo modo de regras, para a demonstração não ficar vazia.

**Limites:** a malha tem 25 aeroportos (mapa médio); o tipo de aeronave é fixo (narrow-body); pedidos fora do tema (preço, hotel, outros assuntos) são recusados. Se a janela não aparecer (só texto), execute a célula de novo com a sessão do Colab conectada.

Esta interface atende ao bônus do enunciado: uma interface de conversa em que o usuário fala com o agente por texto.

In [ ]:
import json
import re
import unicodedata

from IPython.display import Markdown, display

try:
    import ipywidgets as widgets
except ImportError:
    get_ipython().run_line_magic("pip", "install -q ipywidgets")
    import ipywidgets as widgets

# ---------------------------------------------------------------- modo sem LLM
ALIASES = {
    "ABQ": ["albuquerque"], "ATL": ["atlanta"], "BNA": ["nashville"], "BOS": ["boston"],
    "CLE": ["cleveland"], "CLT": ["charlotte"], "DEN": ["denver"], "DFW": ["dallas", "fort worth"],
    "DTW": ["detroit"], "IAH": ["houston"], "IND": ["indianapolis"],
    "JFK": ["nova york", "nova iorque", "new york"], "LAS": ["las vegas"], "LAX": ["los angeles"],
    "MEM": ["memphis"], "MIA": ["miami"], "MSP": ["minneapolis"], "ORD": ["chicago"],
    "PDX": ["portland"], "PHL": ["filadelfia", "philadelphia"], "PHX": ["phoenix", "fenix"],
    "SAN": ["san diego"], "SEA": ["seattle"], "SFO": ["san francisco"],
    "STL": ["st louis", "st. louis", "saint louis"],
}
PALAVRAS_COMUNS = {"ATE", "QUE", "UMA", "COM", "POR", "NAO", "SEM", "MAS", "SAO", "QUAL", "ROTA"}

def _sem_acento(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def _mencoes(T, conhecidos):
    # aeroportos citados (codigo IATA ou nome da cidade): (inicio, fim, codigo, existe_na_malha)
    t = T.lower()
    achados = []
    if not T.isupper():
        for m in re.finditer(r"\b[A-Z]{3}\b", T):
            if m.group() in conhecidos or m.group() not in PALAVRAS_COMUNS:
                achados.append((m.start(), m.end(), m.group(), m.group() in conhecidos))
    else:
        for m in re.finditer(r"\b[A-Z]{3}\b", T):
            if m.group() in conhecidos:
                achados.append((m.start(), m.end(), m.group(), True))
    for cod, nomes in ALIASES.items():
        if cod in conhecidos:
            for nome in nomes:
                for m in re.finditer(r"\b" + re.escape(nome) + r"\b", t):
                    achados.append((m.start(), m.end(), cod, True))
    achados.sort()
    limpos = []
    for a in achados:
        if limpos and a[0] < limpos[-1][1]:
            continue
        if limpos and limpos[-1][2] == a[2] and a[0] - limpos[-1][1] <= 3:   # "Chicago (ORD)"
            continue
        limpos.append(a)
    return limpos

def interpretar_por_regras(texto):
    # devolve (argumentos_da_ferramenta, notas, mensagem_de_erro)
    T = _sem_acento(texto)
    t = T.lower()
    conhecidos = set(tools._aeroportos)
    notas = []

    if re.search(r"ignor\w* (as |suas )?(regras|instruc\w*)|invente|inventar|esqueca", t):
        notas.append("Ignorei o pedido de inventar dados: a rota abaixo vem da ferramenta.")
    if re.search(r"barato|preco|passagem|hotel|reserv\w*|capital d|comprar|alugu?el|alugar", t):
        return None, notas, ("Fora do escopo: só planejo rotas aéreas dentro da malha. "
                             "Não trato de preços, reservas, hotéis ou outros assuntos.")

    ments = _mencoes(T, conhecidos)
    if not ments:
        return None, notas, ("Não identifiquei aeroportos no pedido. Diga a origem e o destino "
                             "(código IATA, como CLT e LAX, ou o nome da cidade).")
    usados, args = set(), {}

    tipo = None
    for padrao, nome in ((r"fechad|interditad|bloquead", "Closure"),
                         (r"acidente|atras", "Delay"), (r"tempestade|tormenta", "Storm")):
        if re.search(padrao, t):
            tipo = nome
            break
    eventos = []
    for m in re.finditer(r"\b(entre|trecho)\b", t):
        prox = [i for i, a in enumerate(ments)
                if 0 <= a[0] - m.end() <= 40 and i not in usados][:2]
        if len(prox) == 2 and tipo:
            eventos.append({"de": ments[prox[0]][2], "para": ments[prox[1]][2], "tipo": tipo})
            usados.update(prox)
    if eventos:
        args["eventos"] = eventos

    evitar = []
    for m in re.finditer(r"evit\w*|sem passar por|nao passar por|nao quero passar por|nao passe por", t):
        resto = t[m.end():]
        fim = re.search(r"[.;?!,]|\bpara\b|\bate\b|\bmas\b|\bquero\b|\bvou\b", resto)
        limite = m.end() + (fim.start() if fim else len(resto))
        janela = [i for i, a in enumerate(ments) if m.end() <= a[0] < limite and i not in usados]
        if not janela and re.search(r"\bla\b|\bali\b|\bnesse\b|\besse\b", t[m.end():limite]):
            janela = [i for i, a in enumerate(ments) if a[1] <= m.start() and i not in usados][-1:]
        for i in janela:
            evitar.append(ments[i][2])
            usados.add(i)
    if evitar:
        args["evitar"] = sorted(set(evitar))

    restantes = [i for i in range(len(ments)) if i not in usados]
    origem = destino = None
    for i in restantes:
        antes = t[max(0, ments[i][0] - 16):ments[i][0]]
        if origem is None and re.search(r"\b(de|desde|saindo de|sair de|partindo de)\s+$", antes):
            origem = i
        elif destino is None and re.search(r"\b(para|pra|ate|a|ao|em|na|no)\s+$", antes):
            destino = i
    sobra = [i for i in restantes if i not in (origem, destino)]
    if origem is None and sobra:
        origem = sobra.pop(0)
    if destino is None and sobra:
        destino = sobra.pop(0)
    if origem is None and destino is None:
        return None, notas, "Não consegui identificar origem e destino."
    if destino is None:
        return None, notas, f"Faltou o destino (origem identificada: {ments[origem][2]}). Para onde você quer ir?"
    if origem is None:
        return None, notas, f"Faltou a origem (destino identificado: {ments[destino][2]}). De onde você sai?"
    args = {"origem": ments[origem][2], "destino": ments[destino][2], **args}

    if re.search(r"fora d[eo] pico|manha cedo|madrugada|off-?peak", t):
        args["horario"] = "Off-peak"
    elif re.search(r"\bpico\b|rush|\bpeak\b", t):
        args["horario"] = "Peak"
    if re.search(r"ceu limpo|tempo limpo|tempo bom|\bclear\b", t):
        args["clima"] = "Clear"
    elif re.search(r"chuv|chov", t):
        args["clima"] = "Rain"
    elif re.search(r"\bneve|nevand", t):
        args["clima"] = "Snow"
    elif "tempestade" in t and not eventos and not evitar:
        args["clima"] = "Storm"
    forte = "forte" in t
    if re.search(r"vento (de )?(proa|contrario|contra)", t):
        args["vento"] = -25 if forte else -15
    elif re.search(r"vento (de )?(cauda|a favor)", t):
        args["vento"] = 25 if forte else 15
    m = re.search(r"congestion\w*\s*(?:de\s*)?(?:nivel|indice)?\s*([1-5])\b", t)
    if m:
        args["congestionamento"] = int(m.group(1))
    elif re.search(r"muito congestionad|congestionamento (alto|intenso|forte)", t):
        args["congestionamento"] = 5
    if re.search(r"turboprop|wide-?body|narrow-?body", t):
        notas.append("O tipo de aeronave não é configurável nesta ferramenta: a busca usa narrow-body.")
    return args, notas, None

def _hm(minutos):
    h, mm = divmod(int(round(minutos)), 60)
    return f"{h} h {mm:02d} min" if h else f"{mm} min"

def _texto_da_ferramenta(r):
    if not r["ok"]:
        return f"**A ferramenta não encontrou rota:** {r['erro']}"
    linhas = [f"**Rota mais rápida:** {' → '.join(r['rota'])}",
              f"- Tempo previsto: **{r['tempo_previsto_min']} min** ({_hm(r['tempo_previsto_min'])}) "
              f"· distância {r['distancia_km']} km"]
    curta = r["rota_mais_curta"]
    if curta:
        linhas.append(f"- Rota mais curta em km: {' → '.join(curta['rota'])} "
                      f"({curta['distancia_km']} km, {curta['tempo_previsto_min']} min)")
        if r["economia_min"] and r["economia_min"] > 0:
            linhas.append(f"- Tempo economizado em relação à rota mais curta: **{r['economia_min']} min**")
        else:
            linhas.append("- A rota mais curta em km já é a mais rápida neste cenário.")
    ctx = r["contexto"]
    linhas.append(f"- Condições consideradas: clima {ctx['clima']}, horário {ctx['horario']}, "
                  f"vento {ctx['vento']} nós, congestionamento {ctx['congestionamento']}"
                  + (f", eventos {ctx['eventos']}" if ctx["eventos"] else ""))
    return "\n".join(linhas)

def consultar(texto, usar_llm):
    # devolve (markdown_da_resposta, chamadas_de_ferramenta, linha_de_rodape)
    aviso = ""
    if usar_llm and globals().get("agent") is not None:
        r = agent.run(texto)
        if not r["erro"] and r["resposta"]:
            rodape = f"{r['latencia_s']} s · {r['tokens_in']}+{r['tokens_out']} tokens · modo LLM"
            return r["resposta"], [(c["nome"], c["entrada"]) for c in r["chamadas"]], rodape
        aviso = (f"> O LLM não respondeu ({r['erro'] or 'resposta vazia'}). "
                 "Respondendo pelo interpretador por regras.\n\n")
    args, notas, erro = interpretar_por_regras(texto)
    rodape = "modo regras (sem LLM)"
    if erro:
        return aviso + erro, [], rodape
    r = tools.buscar_rota(**args)
    corpo = _texto_da_ferramenta(r)
    extra = "".join(f"\n\n> {n}" for n in notas)
    return aviso + corpo + extra, [("buscar_rota", args)], rodape

# ---------------------------------------------------------------- interface
if "pedidos" in globals():
    _exemplos = [p["pedido"] for p in pedidos]
else:
    with open("data/pedidos_teste.json", encoding="utf-8") as f:
        _exemplos = [p["pedido"] for p in json.load(f)]

_llm_ok = bool(globals().get("TEM_LLM")) and globals().get("agent") is not None
modo = widgets.ToggleButtons(
    options=[("LLM", True), ("Regras (sem LLM)", False)] if _llm_ok else [("Regras (sem LLM)", False)],
    value=True if _llm_ok else False, description="Modo:", style={"description_width": "50px"})
estado = widgets.HTML(
    "<b>LLM configurado.</b> O agente interpreta e explica o pedido."
    if _llm_ok else
    "<b>LLM não configurado</b> (Secrets ausentes ou agente indisponível): usando o interpretador "
    "por regras. A busca e o modelo de custo são os mesmos.")
exemplos = widgets.Dropdown(options=[("— escolha um exemplo —", "")] + [(e[:80], e) for e in _exemplos],
                            description="Exemplos:", layout=widgets.Layout(width="100%"))
caixa = widgets.Textarea(placeholder="Escreva seu pedido, por exemplo: Qual a melhor rota de ABQ para STL?",
                         layout=widgets.Layout(width="100%", height="80px"))
enviar = widgets.Button(description="Enviar", button_style="primary", icon="paper-plane")
limpar = widgets.Button(description="Limpar", icon="eraser")
saida = widgets.Output()

def _ao_escolher(mudanca):
    if mudanca["new"]:
        caixa.value = mudanca["new"]
exemplos.observe(_ao_escolher, names="value")

def _ao_enviar(_):
    texto = caixa.value.strip()
    with saida:
        saida.clear_output(wait=True)
        if not texto:
            display(Markdown("Escreva um pedido na caixa de texto."))
            return
        if "tools" not in globals():
            display(Markdown("Execute antes as seções 1 a 11: a interface usa as ferramentas do agente."))
            return
        enviar.disabled = True
        try:
            display(Markdown("*Consultando o agente...*"))
            resposta, chamadas, rodape = consultar(texto, modo.value)
            saida.clear_output(wait=True)
            display(Markdown(f"**Você:** {texto}"))
            display(Markdown(f"**Agente:**\n\n{resposta}"))
            for nome, entrada in chamadas:
                display(Markdown("**Ferramenta chamada:** `" + nome + "`\n\n```json\n"
                                 + json.dumps(entrada, ensure_ascii=False, indent=2) + "\n```"))
            display(Markdown(f"<sub>{rodape}</sub>"))
        except Exception as e:
            saida.clear_output(wait=True)
            display(Markdown(f"**Erro inesperado:** `{type(e).__name__}: {e}`"))
        finally:
            enviar.disabled = False

def _ao_limpar(_):
    caixa.value = ""
    exemplos.value = ""
    saida.clear_output()

enviar.on_click(_ao_enviar)
limpar.on_click(_ao_limpar)

display(widgets.VBox([
    widgets.HTML("<h3>Converse com o agente de rotas aéreas</h3>"),
    estado, modo, exemplos, caixa, widgets.HBox([enviar, limpar]), saida,
]))

## 16. Salvamento dos resultados

Salva as tabelas geradas nesta execução em `results/` (com sufixo `_notebook`, para não sobrescrever os resultados do relatório).

In [ ]:
fase1.to_csv("results/fase1_notebook.csv", index=False)
pd.DataFrame(resumo).to_csv("results/fase1_todos_os_pares_notebook.csv", index=False)
resultados.to_csv("results/fase2_metricas_notebook.csv")
mudancas_df.to_csv("results/fase2_rotas_que_mudam_notebook.csv", index=False)
print("Salvos:", *sorted(p.name for p in Path("results").glob("*_notebook.csv")), sep="\n- ")

## 17. IA responsável

- **Viés dos dados:** os dados são sintéticos e refletem as regras escolhidas pelo grupo (por exemplo, o efeito de tempestade e de congestionamento). O modelo reproduz a simulação, não a operação aérea real. A cobertura é de uma malha pequena (25 aeroportos no mapa médio).
- **Limites do agente:** só planeja rotas dentro da malha; o tipo de aeronave é fixo (narrow-body) na ferramenta; recusa pedidos fora do escopo. Ver seção 14.
- **Alucinação:** mitigada por design (o LLM não calcula rota) e medida pelo critério de fidelidade da resposta.
- **Custo e latência:** ver seções 13 e 14.
- **LGPD:** o projeto **não usa dados pessoais**. Os dados de voo são sintéticos e os pedidos de teste são fictícios. Se o agente fosse usado com dados reais de passageiros, os pedidos enviados ao provedor do LLM passariam por tratamento de dados pessoais (base legal, minimização e contrato com o provedor).

## 18. Conclusão

O agente reúne as três fases em uma peça só: o LLM interpreta o pedido, o A* da Fase 1 busca a rota usando os tempos aprendidos na Fase 2, e o LLM explica o resultado. Nenhum código das fases anteriores foi descartado; cada uma virou componente do agente final.

**PREENCHER (após executar):** 2 a 4 frases com os números observados: ganho do modelo sobre o baseline, em quantos pares a rota mudou, acertos do agente nos pedidos de teste e principal limitação.

## 19. Declaração de uso de ferramentas de IA generativa

**PREENCHER:** descrevam quais ferramentas de IA generativa foram usadas no desenvolvimento (por exemplo, geração de código, revisão, escrita de texto), em quais partes e como o grupo conferiu o resultado. Em seguida, o LLM usado dentro do agente: provedor e modelo informados na seção 3.

## Equipe

- Sofia de Carvalho Brito — 2512130063 — Fase 1 (Formulação e Busca)
- Victor Sousa Pereira — 2512130015 — Fase 2 (Dados e Aprendizado)
- Pedro de Faria Mello — 2512130037 — Fase 3 (Agente com LLM e IA Responsável)